In [20]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import RandomizedSearchCV

import joblib


In [2]:
preprocessor = joblib.load("../models/preprocessor.pkl")

In [6]:
split_data = joblib.load("../models/train_test_split.pkl")

X_train = split_data["X_train"]
X_test = split_data["X_test"]
y_train = split_data["y_train"]
y_test = split_data["y_test"]

In [4]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [7]:
lr_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [8]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10))
])

In [9]:
rf_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

In [10]:
gb_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

### Create a function:

In [13]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)

    rmse = np.sqrt(
        mse
    )

    r2 = r2_score(y_test, predictions)

    return mae, rmse, r2

### Then create a comparison table:

In [18]:
results = []

models = {
    "Linear Regression": lr_pipeline,
    "Ridge": ridge_pipeline,
    "Random Forest": rf_pipeline,
    "Gradient Boosting": gb_pipeline
}

for name, model in models.items():
    model.fit(X_train, y_train)

    mae, rmse, r2 = evaluate_model(
        model,
        X_test,
        y_test
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    results_df = pd.DataFrame(results)

    results_df.sort_values(
        "RMSE"
    )

In [19]:
results_df

,Model,MAE,RMSE,R2
0,Linear Regression,18224.710070,29440.949206,0.886997
1,Ridge,18876.061973,30513.201755,0.878616
2,Random Forest,17505.399053,29324.135705,0.887892
3,Gradient Boosting,16543.858572,26833.369129,0.906128


In [21]:
gb_param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__subsample": [0.8, 0.9, 1.0]
}

In [22]:
gb_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_grid,
    n_iter=30,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [23]:
gb_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,n_iter,30
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [24]:
gb_search.best_params_

{'model__subsample': 1.0,
 'model__n_estimators': 500,
 'model__min_samples_split': 5,
 'model__min_samples_leaf': 2,
 'model__max_depth': 3,
 'model__learning_rate': 0.03}

In [25]:
gb_search.best_score_

np.float64(-27202.44133678574)

In [26]:
best_cv_rmse = -gb_search.best_score_

print("Best CV RMSE:", best_cv_rmse)

Best CV RMSE: 27202.44133678574


In [27]:
best_gb = gb_search.best_estimator_

In [28]:
best_gb

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
y_pred = best_gb.predict(X_test)

In [30]:
y_pred

array([143649.17361315, 350340.02832674, 112209.20233071, 151056.7916925 ,
       336050.13781578,  78855.17133897, 219490.37876709, 144236.94735271,
        78089.57035396, 144663.36902202, 158269.83179186, 126850.03154993,
       112521.53528082, 207929.91144404, 174389.1344504 , 129751.30217292,
       193726.70636829, 136421.61474042, 111683.65656342, 214543.07011921,
       165649.42792903, 220316.28832699, 168467.16245174, 129372.83413745,
       202159.77221699, 159558.61835145, 199006.36555569, 114744.17017365,
       174942.33770072, 190874.54515904, 121290.25180717, 252202.52137903,
       232483.13457503, 114704.82504027, 256945.44518672, 146661.48630452,
       139247.60482679, 206577.31978312, 330288.92999837, 109729.04666428,
       123344.57220515, 244449.89566317, 115768.79779743, 375083.13949027,
       123547.96092432, 133951.08700765, 111593.2125274 , 126787.52697557,
       431738.17508935, 130088.41131376, 121731.69950132, 192920.36914337,
       109007.26863033, 3

In [31]:
joblib.dump(y_pred,"../models/y_pred.pkl")

['../models/y_pred.pkl']